In [1]:
# Imports
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pycaret
np.random.seed(42) #seed global

pd.set_option("display.max_columns", None) #sem limite de colunas mostradas pelo pandas

In [2]:
df_og = pd.read_csv('../data/processed/dinn_scp_dataset.csv')
df = df_og.copy() #criando cópia do dataframe original
 
countries = ['Argentina','Colombia','Uruguay','Peru','Chile']
df = df.copy().query('country in @countries')
df.drop(['dmachexp', 'ict_mach_us_Y1','dict_mach','ict_mach_us_Y2'],axis=1,inplace=True,errors='ignore')
df.drop(['dinnpc','dinnpd','dongoingia', 'dabandia','fucode'],axis=1,inplace=True,errors='ignore')



In [3]:
isics = pd.read_csv('../data/raw/isics.csv')
isics_letras = isics.Code.str.isalpha()
isics = isics[~isics_letras]
isics = isics.loc[isics.Code.apply(len)<=3]
isics.Code = isics.Code.astype(int)
isics.Code = isics.Code.astype(str)
isics = isics.loc[isics.Code.apply(len)<=2]

df.isic3_2d = df.isic3_2d.astype('Int32')
df.isic3_2d = df.isic3_2d.astype('str')

df = df.merge(isics, left_on='isic3_2d',right_on='Code')
df.isic3_2d = df.Description
df.drop('Description',axis=1,inplace=True)

In [4]:
categs_setor_low = df.isic3_1d.value_counts(normalize=True)[df.isic3_1d.value_counts(normalize=True)<0.06].index
df['isic3_1d']=df['isic3_1d'].replace(categs_setor_low,'other')

In [5]:
total_null = df.isnull().sum().sort_values(ascending=False)
percent_null = (total_null/df.shape[0]).sort_values(ascending=False)
print(f'{total_null} \n\n{round(percent_null*100, 2)}')

target = 'dinn_scp'

#df_edit = df.drop(percent_null[percent_null>0.50].index,axis=1)
env_index = df[target].dropna().index
env_df = df.loc[env_index]

env_null = env_df.isnull().sum().sort_values(ascending=False)
percent_env_null = (env_null/env_df.shape[0]).sort_values(ascending=False)

df_env_edit = env_df.drop(percent_env_null[(percent_env_null>0.20) & (percent_env_null != target)].index,axis=1)
df_env_edit.info()

phd_Y3            108836
master_Y3         108836
undergrad_Y3      105184
nontert_Y3        105184
postgrad_Y3       105184
second_Y3         105175
export_us_Y3       94819
sales_us_Y3        84859
unidegree_Y3       84760
empl_Y3            84726
export_us_Y1       82603
sales_us_Y1        72643
degree_agrsc       68146
degree_nesc        63011
degree_et          63011
degree_hum         60055
degree_ssc         60055
degree_medsc       60055
degree_ingnesc     57971
export_us_Y2       44367
sales_us_Y2        34405
dexport            27784
master_Y2          25978
nontert_Y2         25978
postgrad_Y2        25978
phd_Y2             25978
undergrad_Y2       25978
master_Y1          25724
phd_Y1             25724
second_Y2          25193
undergrad_Y1       23585
postgrad_Y1        23585
nontert_Y1         23585
second_Y1          23049
dconexp            21648
isic3_1d           21161
IPexp_us_Y2        19116
IPexp_us_Y1        19116
unidegree_Y1       15219
unidegree_Y2       15127


In [6]:
df.groupby(['country','isic3_1d']).count()#.value_counts(normalize=True)

exp_f   year  isic3_2d  sales_us_Y1  \
country   isic3_1d                                                          
Argentina Manufacturing               10175  10175     10175         9962   
Chile     Manufacturing                3541   3541      3541         3541   
          other                        4089   4089      4089         4089   
Colombia  Manufacturing               47947  47947     47947            0   
          Wholesale and retail trade   9891   9891      9891            0   
          other                        9698   9698      9698            0   
Uruguay   Manufacturing                1334   1334      1334            0   
          other                        1000   1000      1000            0   

                                      sales_us_Y2  sales_us_Y3  export_us_Y1  \
country   isic3_1d                                                             
Argentina Manufacturing                      9962         9962             0   
Chile     Manufacturing                      3541            0          3541   
          other                              4089            0          4089   
Colombia  Manufacturing                     25032            0             0   
          Wholesale and retail trade         7563            0             0   
          other                              7157            0             0   
Uruguay   Manufacturing                         0         1334             0   
          other                                 0         1000             0   

                                      export_us_Y2  export_us_Y3  dexport  \
country   isic3_1d                                                          
Argentina Manufacturing                          0             0    10175   
Chile     Manufacturing                       3541             0     3541   
          other                               4089             0     4089   
Colombia  Manufacturing                      25032             0    25032   
          Wholesale and retail trade          7563             0     7563   
          other                               7157             0     7157   
Uruguay   Manufacturing                          0          1334     1334   
          other                                  0          1000     1000   

                                      empl_Y1  empl_Y2  empl_Y3  phd_Y1  \
country   isic3_1d                                                        
Argentina Manufacturing                 10096    10096    10096       0   
Chile     Manufacturing                  3541     3541        0    2523   
          other                          4089     4089        0    3877   
Colombia  Manufacturing                 47945    47945        0   47760   
          Wholesale and retail trade     9891     9891        0    9872   
          other                          9698     9698        0    9610   
Uruguay   Manufacturing                     0        0     1334       0   
          other                             0        0     1000       0   

                                      phd_Y2  phd_Y3  master_Y1  master_Y2  \
country   isic3_1d                                                           
Argentina Manufacturing                    0       0          0          0   
Chile     Manufacturing                 2513       0       2523       2513   
          other                         3862       0       3877       3862   
Colombia  Manufacturing                47653       0      47760      47653   
          Wholesale and retail trade    9843       0       9872       9843   
          other                         9593       0       9610       9593   
Uruguay   Manufacturing                    0       0          0          0   
          other                            0       0          0          0   

                                      master_Y3  postgrad_Y1  postgrad_Y2  \
country   isic3_1d                                                          
Argentina

In [7]:
df_env_edit.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 108836 entries, 0 to 108835
Data columns (total 23 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   country       108836 non-null  object 
 1   exp_f         108836 non-null  float64
 2   year          108836 non-null  int64  
 3   isic3_1d      87675 non-null   object 
 4   isic3_2d      108836 non-null  object 
 5   empl_Y1       103860 non-null  float64
 6   empl_Y2       100203 non-null  float64
 7   unidegree_Y1  93617 non-null   float64
 8   unidegree_Y2  93709 non-null   float64
 9   RD_empl       101602 non-null  float64
 10  IPexp_us_Y1   89720 non-null   float64
 11  IPexp_us_Y2   89720 non-null   float64
 12  drdintexp     108836 non-null  int64  
 13  drdextexp     108836 non-null  int64  
 14  dictexp       101206 non-null  float64
 15  dTTexp        98479 non-null   float64
 16  dconexp       87188 non-null   float64
 17  dIPexp        97545 non-null   float64
 18  dtra

In [8]:
from pycaret import classification as clf

exp_name = clf.setup(data = df_env_edit,  target = target,
                     remove_multicollinearity=True,
                     ignore_low_variance=True,
                     feature_selection=True,
                     fix_imbalance=True)

,Description,Value
0,session_id,6951
1,Target,dinn_scp
2,Target Type,Binary
3,Label Encoded,"0.0: 0, 1.0: 1"
4,Original Data,"(108836, 23)"
5,Missing Values,True
6,Numeric Features,9
7,Categorical Features,13
8,Ordinal Features,False
9,High Cardinality Features,False


In [9]:
best_model = clf.compare_models(n_select=6)#include=['lightgbm','ridge','dt','ada',])#, exclude=['gbc'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9078,0.9172,0.5170,0.5932,0.5523,0.5012,0.5028,2.6060
rf,Random Forest Classifier,0.9039,0.9006,0.4954,0.5732,0.5314,0.4782,0.4798,12.5940
gbc,Gradient Boosting Classifier,0.8968,0.9087,0.6072,0.5269,0.5641,0.5059,0.5076,31.5620
et,Extra Trees Classifier,0.8966,0.8542,0.4684,0.5343,0.4991,0.4418,0.4430,21.1490
ada,Ada Boost Classifier,0.8952,0.8950,0.5747,0.5215,0.5466,0.4875,0.4884,5.9900
dt,Decision Tree Classifier,0.8918,0.7257,0.4898,0.5090,0.4991,0.4385,0.4387,1.6380
dummy,Dummy Classifier,0.8900,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.6670
nb,Naive Bayes,0.8808,0.6613,0.0705,0.3147,0.1152,0.0780,0.1039,0.6700
knn,K Neighbors Classifier,0.8338,0.8348,0.7091,0.3678,0.4842,0.3968,0.4273,24.0840
ridge,Ridge Classifier,0.8268,0.0000,0.8206,0.3706,0.5105,0.4231,0.4725,0.8010


In [13]:
clf.tune_model(best_model[0])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9083,0.9129,0.4761,0.6055,0.5331,0.4830,0.4873
1,0.9093,0.9226,0.5179,0.6019,0.5568,0.5066,0.5083
2,0.9060,0.9187,0.5042,0.5851,0.5416,0.4896,0.4913
3,0.9111,0.9151,0.5089,0.6171,0.5578,0.5089,0.5118
4,0.9094,0.9157,0.4887,0.6110,0.5430,0.4935,0.4973
5,0.9092,0.9161,0.5036,0.6046,0.5495,0.4994,0.5020
6,0.9069,0.9124,0.4893,0.5933,0.5363,0.4851,0.4879
7,0.9047,0.9090,0.4690,0.5831,0.5198,0.4676,0.4710
8,0.9069,0.9082,0.4988,0.5912,0.5411,0.4897,0.4919


LGBMClassifier(bagging_fraction=0.6, bagging_freq=1, boosting_type='gbdt',
               class_weight=None, colsample_bytree=1.0, feature_fraction=0.7,
               importance_type='split', learning_rate=0.5, max_depth=-1,
               min_child_samples=1, min_child_weight=0.001, min_split_gain=0.1,
               n_estimators=50, n_jobs=-1, num_leaves=90, objective=None,
               random_state=6951, reg_alpha=1, reg_lambda=0.001, silent='warn',
               subsample=1.0, subsample_for_bin=200000, subsample_freq=0)

In [14]:
clf.evaluate_model(best_model[0])


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Hyperparameters', 'param…

In [16]:
original = clf.get_config('data_before_preprocess')
X_test = clf.get_config('X_test')
actual = clf.get_config('y_test')
preds = best_model[0].predict(X_test)

In [17]:
y_train = clf.get_config('y_train')

In [69]:
category = 'dtrainexp'
per_cat_results = pd.concat([original.loc[X_test.index,category].reset_index(drop=True),actual.reset_index(drop=True),pd.Series(preds)],axis=1,ignore_index=True,)
per_cat_results.columns = [category,'actual','predicted']

In [70]:
from sklearn.metrics import precision_score,recall_score,f1_score
results = []
index = per_cat_results[category].unique().tolist()
columns=['precision','recall','f-score']
for ctry in per_cat_results[category].unique():
    df_metrics = per_cat_results.query(f'{category} == @ctry')
    p = precision_score(df_metrics.actual,df_metrics.predicted)
    r = recall_score(df_metrics.actual,df_metrics.predicted)
    f = f1_score(df_metrics.actual,df_metrics.predicted)
    results.append([p,r,f])

metrics = pd.DataFrame(results,index=index,columns=columns)

In [71]:
df_env_edit.columns

Index(['country', 'exp_f', 'year', 'isic3_1d', 'isic3_2d', 'empl_Y1',
       'empl_Y2', 'unidegree_Y1', 'unidegree_Y2', 'RD_empl', 'IPexp_us_Y1',
       'IPexp_us_Y2', 'drdintexp', 'drdextexp', 'dictexp', 'dTTexp', 'dconexp',
       'dIPexp', 'dtrainexp', 'dIDexp', 'dmktexp', 'dinn_scp', 'Code'],
      dtype='object')

In [72]:
metrics.head(15)

,precision,recall,f-score
0,0.600482,0.369254,0.457300
1,0.604883,0.734997,0.663623


In [29]:
clf.get_config('prep_pipe')

Pipeline(memory=None,
         steps=[('dtypes',
                 DataTypes_Auto_infer(categorical_features=[],
                                      display_types=True, features_todrop=[],
                                      id_columns=[],
                                      ml_usecase='classification',
                                      numerical_features=[], target='dinn',
                                      time_features=[])),
                ('imputer',
                 Simple_Imputer(categorical_strategy='not_available',
                                fill_value_categorical=None,
                                fill_value_numerical=None,
                                numeric_strateg...
                 Advanced_Feature_Selection_Classic(ml_usecase='classification',
                                                    n_jobs=-1,
                                                    random_state=8191,
                                                    subclass='binary',
 